# Compute Robustness

In [1]:
# Scumi annotated labels

import anndata as ad
import scanpy as sc
from sklearn.linear_model import LogisticRegression
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from custom_stopper import CustomStopper
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
Before filtering highly variable genes 

In [2]:
import warnings
warnings.filterwarnings("ignore")  # Optional: reduziert Ausgaben

## LogisticRegression

In [2]:
# Robustness Tests with Logging and no filter warnings
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/logisticregression_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, 'scumi-annotation', '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

2026-06-30 22:15:52,964 - INFO - --- In distribution testset ---
2026-06-30 22:15:53,049 - INFO - Baseline accuracy score 0.9213
2026-06-30 22:15:53,108 - INFO - Classification Report:
                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.87      1.00      0.93      3910
   Cytotoxic T cell       0.98      0.62      0.76      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.88      0.96      0.92       791
        Plasma cell       1.00      0.98      0.99        49

           accuracy                           0.92      9281
          macro avg       0.97      0.89      0.92      9281
       weighted avg       0.93      0.92      0.92      9281



/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


2026-06-30 22:15:54,858 - INFO - Random dropout accuracy score 0.9028
2026-06-30 22:15:55,277 - INFO - Total samples: 9281
2026-06-30 22:15:55,277 - INFO - Number of inconsistent predictions: 0
2026-06-30 22:15:55,455 - INFO - Feature importance dropout (0.1% features dropped) accuracy score 0.9192
2026-06-30 22:15:55,631 - INFO - Feature importance dropout (0.5% features dropped) accuracy score 0.8443
2026-06-30 22:15:55,803 - INFO - Feature importance dropout (1.0% features dropped) accuracy score 0.7861
2026-06-30 22:15:55,975 - INFO - Feature importance dropout (2.0% features dropped) accuracy score 0.7327
2026-06-30 22:15:55,977 - INFO - --- Out of data distribution ---
2026-06-30 22:17:51,470 - INFO - Genes expected in training set: 10000
2026-06-30 22:17:51,471 - INFO - Genes actually matched in test set: 8693
2026-06-30 22:17:51,486 - INFO - Training data Max-Value: 8.63405704498291
2026-06-30 22:17:51,565 - INFO - Test data Max-Value: 8.726716041564941
2026-06-30 22:17:51,957 

In [3]:
# Robustness Tests with Logging and no filter warnings
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/logisticregression_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, 'scumi-annotation', None, feature_importance, gene_names_test)

2026-06-30 22:18:09,745 - INFO - --- In distribution testset ---
2026-06-30 22:18:09,821 - INFO - Baseline accuracy score 0.9213
2026-06-30 22:18:09,866 - INFO - Classification Report:
                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.87      1.00      0.93      3910
   Cytotoxic T cell       0.98      0.62      0.76      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.88      0.96      0.92       791
        Plasma cell       1.00      0.98      0.99        49

           accuracy                           0.92      9281
          macro avg       0.97      0.89      0.92      9281
       weighted avg       0.93      0.92      0.92      9281

2026-06-30 22:18:11,616 - INFO - Random dropout accuracy score 0.9075
2026-06-30

2026-06-30 22:18:12,718 - ERROR - No out-of-distribution dataset path provided. Skipping out-of-distribution tests.


## Random Forest

In [5]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/randomforest_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8868

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.94      1.00      0.97      3910
   Cytotoxic T cell       1.00      0.44      0.61      1824
     Dendritic cell       1.00      0.20      0.33         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.50      0.99      0.66       791
        Plasma cell       1.00      0.86      0.92        49

           accuracy                           0.89      9281
          macro avg       0.93      0.81      0.81      9281
       weighted avg       0.93      0.89      0.88      9281

Random dropout accuracy score 0.8582
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8429
Feature importance dropout (0.5% 

## ExtraTrees

In [6]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/extratrees_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8877

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.91      1.00      0.95      3910
   Cytotoxic T cell       1.00      0.45      0.62      1824
     Dendritic cell       1.00      0.20      0.33         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.54      0.99      0.70       791
        Plasma cell       1.00      0.86      0.92        49

           accuracy                           0.89      9281
          macro avg       0.93      0.81      0.82      9281
       weighted avg       0.92      0.89      0.88      9281

Random dropout accuracy score 0.8728
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8548
Feature importance dropout (0.5% 

## XGBoost

In [10]:
import pandas as pd

y_train_series = pd.Series(y_train)

min_samples = 5
class_counts = y_train_series.value_counts()
valid_classes = class_counts[class_counts >= min_samples].index

train_mask = y_train_series.isin(valid_classes).values
X_train = X_train[train_mask]
y_train = y_train_series[train_mask]

In [11]:
from sklearn.preprocessing import LabelEncoder

## Encode Labels
le_train = LabelEncoder()
y_train_enc = le_train.fit_transform(y_train.to_numpy() if hasattr(y_train, 'to_numpy') else y_train)

classes_in_train = set(y_train)
mask = y_test.isin(classes_in_train).values

X_test_filtered = X_test[mask]
y_test_filtered = y_test[mask]

y_test_enc = le_train.transform(y_test_filtered.to_numpy() if hasattr(y_test_filtered, 'to_numpy') else y_test_filtered)
X_test = X_test_filtered.to_numpy() if hasattr(X_test_filtered, 'to_numpy') else X_test_filtered

num_classes = len(le_train.classes_)

In [6]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/xgboost_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test_filtered, y_test_enc, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.5297

              precision    recall  f1-score   support

           0       1.00      0.86      0.92       120
           1       1.00      0.01      0.02      2575
           2       1.00      0.76      0.86      3910
           3       0.58      0.58      0.58      1824
           4       0.33      0.20      0.25         5
           5       1.00      0.71      0.83         7
           6       0.89      0.87      0.88       791
           7       0.01      1.00      0.03        49

    accuracy                           0.53      9281
   macro avg       0.73      0.62      0.55      9281
weighted avg       0.90      0.53      0.57      9281

Random dropout accuracy score 0.4315
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.5267
Feature importance dropout (0.5% features dropped) accuracy score 0.4381
Feature importance dropout (1.0% features dr

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [5]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/xgboost_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test_filtered, y_test_enc, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8962

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       120
           1       1.00      1.00      1.00      2575
           2       0.89      1.00      0.94      3910
           3       0.99      0.49      0.65      1824
           4       1.00      0.40      0.57         5
           5       1.00      0.86      0.92         7
           6       0.64      0.99      0.78       791
           7       1.00      0.92      0.96        49

    accuracy                           0.90      9281
   macro avg       0.94      0.83      0.85      9281
weighted avg       0.92      0.90      0.89      9281

Random dropout accuracy score 0.8961
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8692
Feature importance dropout (0.5% features dropped) accuracy score 0.8308
Feature importance dropout (1.0% features dr

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [5]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness_xgboost import test_robustness


with open("models/xgboost_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test_filtered, y_test_enc, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8962

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       120
           1       1.00      1.00      1.00      2575
           2       0.89      1.00      0.94      3910
           3       0.99      0.49      0.65      1824
           4       1.00      0.40      0.57         5
           5       1.00      0.86      0.92         7
           6       0.64      0.99      0.78       791
           7       1.00      0.92      0.96        49

    accuracy                           0.90      9281
   macro avg       0.94      0.83      0.85      9281
weighted avg       0.92      0.90      0.89      9281

Random dropout accuracy score 0.8791
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8692
Feature importance dropout (0.5% features dropped) accuracy score 0.8308
Feature importance dropout (1.0% features dr

ValueError: Fehler: Nach dem Filtern sind 0 Zellen übrig! Prüfe die 'train_classes'.

In [12]:
# scumi annotated; no normalization; adata.X
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/xgboost_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test_filtered, y_test_enc, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.9068

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       120
           1       1.00      1.00      1.00      2575
           2       0.88      1.00      0.94      3910
           3       0.99      0.54      0.70      1824
           4       1.00      0.40      0.57         5
           5       1.00      1.00      1.00         7
           6       0.70      0.99      0.82       791
           7       1.00      0.90      0.95        49

    accuracy                           0.91      9281
   macro avg       0.95      0.85      0.87      9281
weighted avg       0.92      0.91      0.90      9281

Random dropout accuracy score 0.8842
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8780
Feature importance dropout (0.5% features dropped) accuracy score 0.8334
Feature importance dropout (1.0% features dr

AttributeError: 'numpy.ndarray' object has no attribute 'unique'

## LinearSVC

In [7]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/linearsvc_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.9244

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       0.99      1.00      1.00      2575
        CD4+ T cell       0.89      1.00      0.94      3910
   Cytotoxic T cell       0.97      0.65      0.78      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.82      0.95      0.88       791
        Plasma cell       1.00      1.00      1.00        49

           accuracy                           0.92      9281
          macro avg       0.96      0.90      0.92      9281
       weighted avg       0.93      0.92      0.92      9281

Random dropout accuracy score 0.9244
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.9238
Feature importance dropout (0.5% 

## LightGBM

In [8]:
# scumi annotated; no normalization
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/lightgbm_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)
    model.set_params(verbosity=-1)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8501

                     precision    recall  f1-score   support

             B cell       1.00      0.93      0.96       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.87      1.00      0.93      3910
   Cytotoxic T cell       0.98      0.26      0.41      1824
     Dendritic cell       1.00      0.40      0.57         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.49      0.99      0.66       791
        Plasma cell       1.00      0.92      0.96        49

           accuracy                           0.85      9281
          macro avg       0.92      0.81      0.81      9281
       weighted avg       0.90      0.85      0.82      9281

Random dropout accuracy score 0.8427
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8425
Feature importance dropout (0.5% 

## VotingClassifier

In [9]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/votingclassifier_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.8920

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.91      1.00      0.95      3910
   Cytotoxic T cell       1.00      0.47      0.64      1824
     Dendritic cell       1.00      0.20      0.33         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.57      0.99      0.72       791
        Plasma cell       1.00      0.86      0.92        49

           accuracy                           0.89      9281
          macro avg       0.93      0.81      0.82      9281
       weighted avg       0.92      0.89      0.88      9281

Random dropout accuracy score 0.8743
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.8553
Feature importance dropout (0.5% 

## StackingClassifier

In [2]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import compute_model_score_and_robustness


with open("models/stackingclassifier.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/feature_importance_logisticregression.pkl", "rb") as f:
    feature_importance = pickle.load(f)

feature_importance = feature_importance.sort_values('Importance', ascending=False)

compute_model_score_and_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design.h5ad', feature_importance)

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Baseline accuracy score 0.9013


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/pyt

Random dropout accuracy score 0.8897


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/pyt

...

Feature importance dropout (1% features dropped) accuracy score 0.5918


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Feature importance dropout (2% features dropped) accuracy score 0.5346
Out of data distribution
Genes expected in training set: 10000
Genes actually matched in test set: 8408
Training data Max-Value: 2.6092522144317627
Test data Max-Value: 3.0608508586883545


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Baseline accuracy score 0.1664


In [4]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import compute_model_score_and_robustness


with open("models/stackingclassifier.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/feature_importance_logisticregression.pkl", "rb") as f:
    feature_importance = pickle.load(f)

feature_importance = feature_importance.sort_values('Importance', ascending=False)

compute_model_score_and_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated.h5ad', feature_importance)

Baseline accuracy score 0.9013
Random dropout accuracy score 0.8903
Total samples: 9295
Number of inconsistent predictions: 0
Feature importance dropout (0% features dropped) accuracy score 0.8912
Feature importance dropout (0% features dropped) accuracy score 0.6880
Feature importance dropout (1% features dropped) accuracy score 0.5918
Feature importance dropout (2% features dropped) accuracy score 0.5346
Out of data distribution
Genes expected in training set: 10000
Genes actually matched in test set: 8408
Training data Max-Value: 2.6092522
Test data Max-Value: 1.5899227857589722
Baseline accuracy score 0.0000


## Custom Ensemble

In [3]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/custom_ensemble_lr_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.9126

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       0.99      1.00      1.00      2575
        CD4+ T cell       0.86      1.00      0.92      3910
   Cytotoxic T cell       0.97      0.59      0.73      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.86      0.95      0.90       791
        Plasma cell       1.00      1.00      1.00        49

           accuracy                           0.91      9281
          macro avg       0.96      0.89      0.91      9281
       weighted avg       0.92      0.91      0.91      9281

Random dropout accuracy score 0.9044
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.9108
Feature importance dropout (0.5% 

In [10]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/custom_ensemble_linsvc_hard_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.9119

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       0.99      1.00      0.99      2575
        CD4+ T cell       0.87      1.00      0.93      3910
   Cytotoxic T cell       0.95      0.60      0.74      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.83      0.91      0.87       791
        Plasma cell       1.00      1.00      1.00        49

           accuracy                           0.91      9281
          macro avg       0.95      0.89      0.91      9281
       weighted avg       0.92      0.91      0.91      9281

Random dropout accuracy score 0.9016
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.9105
Feature importance dropout (0.5% 

In [11]:
import os
import sys
import pickle

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_path not in sys.path:
    sys.path.append(project_path)

from evaluation.test_robustness import test_robustness


with open("models/custom_ensemble_linsvc_soft_scumi_annotated.pkl", "rb") as f:
    model = pickle.load(f)

with open("../evaluation/master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

#feature_importance = feature_importance.sort_values('Importance', ascending=False)

test_robustness(model, X_test, y_test, '../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad', feature_importance, gene_names_test)

--- In distribution testset ---
Baseline accuracy score 0.9359

                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.91      1.00      0.95      3910
   Cytotoxic T cell       0.93      0.74      0.83      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.89      0.87      0.88       791
        Plasma cell       1.00      1.00      1.00        49

           accuracy                           0.94      9281
          macro avg       0.96      0.90      0.92      9281
       weighted avg       0.94      0.94      0.93      9281

Random dropout accuracy score 0.9330
Total samples: 9281
Number of inconsistent predictions: 0
Feature importance dropout (0.1% features dropped) accuracy score 0.9359
Feature importance dropout (0.5% 